# Ecuador Desigual 🇪🇨 — 02. Abastecimiento de agua

**Fase I · Provincia de El Oro · Censo de Población y Vivienda 2022 (INEC)**

## Propósito
Comparar, a escala cantonal, dos **indicadores de fuente de abastecimiento**:

1. **Municipal (%)**: viviendas cuya fuente es empresa pública/municipio sobre viviendas particulares ocupadas.
2. **Organizado (%)**: viviendas cuya fuente es empresa pública/municipio **o** juntas de agua/organizaciones comunitarias/GAD sobre viviendas particulares ocupadas.

Ambos indicadores describen la **fuente declarada**, **no** garantizan potabilidad, calidad, continuidad del suministro ni disponibilidad de agua por tubería dentro de la vivienda.

**Pregunta exploratoria:** ¿Cómo cambia la comparación entre cantones al incluir las fuentes comunitarias?

## 1. Carga de datos

El archivo `2.1.csv` contiene metadatos antes de los registros. Conservamos una copia sin procesar y verificamos que existe en la ruta esperada.

In [ ]:
from pathlib import Path
import pandas as pd

ruta_csv = Path("../data/raw/archivos/2.1.csv")
if not ruta_csv.exists():
    raise FileNotFoundError(f"No se encontró el CSV en: {ruta_csv.resolve()}")

df_agua = pd.read_csv(ruta_csv, encoding="cp1252", header=None)
print("Dimensiones originales:", df_agua.shape)
df_agua.iloc[7:17, 1:11]

## 2. Comprensión de la estructura

Según los encabezados del tabulado 2.1, desde la fila **11** comienzan los registros. Columnas relevantes:

| Columna | Significado |
|---|---|
| `1` | Provincia |
| `2` | Cantón |
| `3` | Área / total territorial |
| `4` | Forma de recepción del agua / total correspondiente |
| `5` | Total de viviendas particulares ocupadas |
| `6` | Empresa pública / municipio |
| `7` | Juntas de agua / organizaciones comunitarias / GAD |

**Atención:** es una tabla cruzada. Si sumamos registros de *total* y *formas de recepción*, contaríamos las mismas viviendas varias veces.

## 3. Selección de los 14 totales cantonales de El Oro

Conservamos solo filas donde tanto el nivel territorial como la recepción identifican el total del cantón. Así excluimos los totales provinciales y las desagregaciones por recepción.

In [ ]:
df_agua_datos = df_agua.iloc[11:].copy()
df_agua_el_oro = df_agua_datos.loc[df_agua_datos[1] == "El Oro"].copy()

es_total_territorial = df_agua_el_oro[3] == ("Total " + df_agua_el_oro[2])
es_total_recepcion = df_agua_el_oro[4] == ("Total " + df_agua_el_oro[2])

df_agua_cantones = df_agua_el_oro.loc[
    es_total_territorial & es_total_recepcion,
    [2, 5, 6, 7]
].copy()

df_agua_cantones = df_agua_cantones.rename(columns={
    2: "canton",
    5: "total_viviendas",
    6: "viviendas_municipio",
    7: "viviendas_juntas_agua",
})

df_agua_cantones.reset_index(drop=True, inplace=True)
print("Registros:", len(df_agua_cantones))
df_agua_cantones.head()

## 4. Limpieza numérica y validaciones

La coma en estos CSV es separador de miles. La eliminamos antes de convertir a enteros; un valor no convertible provoca un error en lugar de sustituirse silenciosamente.

Comprobamos que existe una fila por cantón, que los conteos son no negativos y que la suma de las dos categorías elegidas no supera el total de viviendas.

In [ ]:
cols_numericas = ["total_viviendas", "viviendas_municipio", "viviendas_juntas_agua"]
for col in cols_numericas:
    df_agua_cantones[col] = pd.to_numeric(
        df_agua_cantones[col].astype("string").str.replace(",", "", regex=False),
        errors="raise"
    )

assert len(df_agua_cantones) == 14, "Se esperaban 14 registros cantonales."
assert df_agua_cantones["canton"].nunique() == 14, "Existen cantones duplicados o faltantes."
assert df_agua_cantones[cols_numericas].notna().all().all(), "Hay valores numéricos faltantes."
assert (df_agua_cantones[cols_numericas] >= 0).all().all(), "Hay conteos negativos."
assert (df_agua_cantones["total_viviendas"] > 0).all(), "Hay denominadores iguales a cero."
assert (
    df_agua_cantones["viviendas_municipio"] + df_agua_cantones["viviendas_juntas_agua"]
    <= df_agua_cantones["total_viviendas"]
).all(), "Las categorías exceden el total de viviendas."

print("Validación básica: correcta")
df_agua_cantones.dtypes

## 5. Indicadores de fuente de abastecimiento

Las categorías de las columnas 6 y 7 son distintas dentro del tabulado, por lo que se pueden agregar para obtener una medida **más amplia de fuente organizada**.

\[
C_{	ext{municipal}} = 100
rac{V_{	ext{municipio}}}{V_{	ext{total}}},\qquad
C_{	ext{organizado}} = 100
rac{V_{	ext{municipio}}+V_{	ext{juntas}}}{V_{	ext{total}}}
\]

Calculamos con precisión completa; el redondeo se aplica **solo al mostrar resultados**.

In [ ]:
df_agua_cantones["cobertura_municipal_pct"] = (
    100 * df_agua_cantones["viviendas_municipio"]
    / df_agua_cantones["total_viviendas"]
)
df_agua_cantones["abastecimiento_organizado_pct"] = (
    100 * (
        df_agua_cantones["viviendas_municipio"]
        + df_agua_cantones["viviendas_juntas_agua"]
    ) / df_agua_cantones["total_viviendas"]
)

df_agua_cantones[[
    "canton", "total_viviendas", "viviendas_municipio", "viviendas_juntas_agua",
    "cobertura_municipal_pct", "abastecimiento_organizado_pct"
]].round(2)

## 6. Comparación cantonal

Primero ordenamos por el indicador municipal. Luego comprobamos cómo cambia el orden si incluimos las fuentes comunitarias.

In [ ]:
columnas_comparacion = [
    "canton", "cobertura_municipal_pct", "abastecimiento_organizado_pct"
]

df_agua_cantones[columnas_comparacion].sort_values(
    "cobertura_municipal_pct"
).round(2)

In [ ]:
df_agua_cantones[columnas_comparacion].sort_values(
    "abastecimiento_organizado_pct"
).round(2)

### Ejercicio de interpretación

- ¿Qué cantón tiene la menor proporción municipal? ¿Es el mismo cuando añadimos juntas de agua?
- ¿Qué cantones cambian notablemente de posición?
- ¿Qué información falta para hablar de acceso a **agua segura** o de calidad del servicio?

**No concluyas que una menor participación municipal es automáticamente mayor vulnerabilidad.**

## 7. Visualización exploratoria (opcional)

Las barras comparan dos porcentajes por cantón. El segundo **incluye** al primero, por lo que **no son categorías mutuamente excluyentes entre sí**.

In [ ]:
import matplotlib.pyplot as plt

ordenados = df_agua_cantones.sort_values("abastecimiento_organizado_pct")
ax = ordenados.plot.barh(
    x="canton",
    y=["cobertura_municipal_pct", "abastecimiento_organizado_pct"],
    figsize=(10, 7),
    width=0.8
)
ax.set_xlabel("Viviendas particulares ocupadas (%)")
ax.set_ylabel("Cantón")
ax.set_title("El Oro, 2022: fuentes de abastecimiento de agua")
ax.set_xlim(0, 100)
ax.legend(["Empresa pública/municipio", "Municipio + juntas/comunitarias"])
plt.tight_layout()
plt.show()

## 8. Validación adicional y siguientes pasos

Antes de integrar este indicador con el de alcantarillado:

1. Comparar los totales de viviendas cantonales con los del tabulado `3.1.csv` (misma unidad y año).
2. Analizar las filas **urbana y rural** de `2.1.csv`, filtrando **también** la columna de forma de recepción para no duplicar registros.
3. Integrar los indicadores en un dataset cantonal de El Oro y documentar explícitamente sus definiciones.

**Limitación:** este notebook aún **no** estudia desigualdades urbano-rurales de agua ni demuestra potabilidad del suministro.

**Fuente:** INEC, VIII Censo de Población y VII de Vivienda (2022), tabulado `2.1.csv` del conjunto de servicios básicos de vivienda.